# 感情AI: 内部信号と感情語の接続実験 — 速度実測プローブ(Colab / T4版)

計算環境をM2 MacBook Air/MLXからGoogle Colab無料枠(T4 GPU 16GB)に変更した(2026-09-10)。
モデルもQwen2.5-0.5B-InstructからQwen2.5-1.5B-Instruct(fp16)に変更した。
理由は事前登録 `事前登録_内部信号と感情語の接続実験.md` の追記欄「2026-09-10」を参照。

このノートブックは**学習を一切行わない**。`TorchPolicy`(`torch_qwen_policy.py`)を
`GroundingEnv` に接続し、20エピソード分の速度・正答率(課題の種類ごとの内訳を含む)・
書式不履行率・エントロピー分布・不正解になった応答の実例を測定して、結果をJSONとして
Google Driveに保存するだけ。

実行順序: 1) GPU確認 → 2) 依存インストール → 3) GitHubからclone → 4) Google Driveマウント →
5) 動作確認(1回だけ生成) → 6) 本番実行(20エピソード)・Driveへ保存 → 7) 結果の確認表示


## 1. GPU確認

ランタイムのタイプが「T4 GPU」になっていることを確認する。

In [ ]:
!nvidia-smi
import torch
print("CUDA利用可能:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "GPUが見つからない。上部メニューの「ランタイム」→「ランタイムのタイプを変更」で"
        "ハードウェアアクセラレータを「T4 GPU」に設定してから、このセルからやり直すこと。"
    )


## 2. 依存のインストール

torchはColabに元から入っているCUDA対応版をそのまま使う(再インストールしない)。
transformers/accelerateを追加インストールする。peftはこの推論のみのスクリプトでは未使用だが、事前登録の「系」にある次段階(LoRAでの学習)に備えて合わせて入れておく。

In [ ]:
!pip install -q "transformers>=4.46" "peft>=0.13" "accelerate>=1.0"


## 3. GitHubからclone

`seina369/homeostatic-agent-experiments`(2026-09-10時点で認証なしアクセスを`git ls-remote`で確認済み・公開リポジトリ)。非公開に変更した場合は、Colabのシークレット(鍵アイコン)に`GITHUB_TOKEN`を登録し、下のREPO_URLを`https://<token>@github.com/...`の形に書き換えること。

In [ ]:
import os

REPO_URL = "https://github.com/seina369/homeostatic-agent-experiments.git"
REPO_DIR = "/content/homeostatic-agent-experiments"

if os.path.isdir(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print("作業ディレクトリ:", os.getcwd())
# 2026-09-12のリポジトリ構成整理により、LLM接続実験一式は llm_grounding/ 配下。
LLM_DIR = os.path.join(REPO_DIR, "llm_grounding")
!ls {LLM_DIR}/torch_qwen_policy.py {LLM_DIR}/run_torch_speed_probe.py {LLM_DIR}/emotion_grounding_env.py


## 4. Google Driveをマウント

結果JSONの保存先を用意する。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUT_DIR = "/content/drive/MyDrive/EmotionalAI"
os.makedirs(DRIVE_OUT_DIR, exist_ok=True)
print("保存先:", DRIVE_OUT_DIR)


## 5. 動作確認(1回だけ生成)

20エピソード分待つ前に、モデルが読み込めて`respond()`が1回動くかだけ軽く確認する。

In [ ]:
import sys
sys.path.insert(0, LLM_DIR)
from torch_qwen_policy import TorchPolicy

_probe_policy = TorchPolicy(verbose=True)
_r = _probe_policy.respond(
    "Task: What is 23 + 19?\nb=450 e=0 u=1.00\n"
    "Reply in one or two sentences, then give the answer as \"A: <answer>\"."
)
print()
print(f"n_tokens={_r.n_tokens} mean_entropy={_r.mean_entropy:.4f}")
del _probe_policy, _r
torch.cuda.empty_cache()


## 6. 本番実行(20エピソード)・Driveへ保存

In [ ]:
import time

OUT_PATH = f"{DRIVE_OUT_DIR}/torch_speed_probe_results.json"

t0 = time.time()
!python3 {LLM_DIR}/run_torch_speed_probe.py --policy torch --episodes 20 --out "{OUT_PATH}"
print(f"所要時間: {time.time() - t0:.1f}秒")


## 7. 結果の確認表示

In [ ]:
import json

with open(OUT_PATH, encoding="utf-8") as f:
    result = json.load(f)

print("policy:", result["policy"], "/ model:", result["model"])
print(f"episode_seconds_mean={result['episode_seconds_mean']:.2f}s +/- {result['episode_seconds_std']:.2f}s")
print(f"correct_rate={result['correct_rate']:.3f}  format_fail_rate={result['format_fail_rate']:.3f}")
print("accuracy_by_kind:")
for k, s in sorted(result["accuracy_by_kind"].items()):
    print(f"  {k}: {s['accuracy']:.3f} ({s['correct']}/{s['n']})")
print(f"entropy: mean={result['entropy_mean']:.3f} p10={result['entropy_p10']:.3f} "
      f"p50={result['entropy_p50']:.3f} p90={result['entropy_p90']:.3f} max={result['entropy_max']:.3f}")
print(f"mean_tokens_per_response={result['mean_tokens_per_response']:.1f}")
print()
print("不正解の実例:")
for ex in result["incorrect_examples"]:
    print(f"- [{ex['kind']}] {ex['task_prompt']} (正解: {ex['correct_answer']})")
    print(f"    抽出answer: {ex['extracted_answer']}  entropy={ex['entropy']:.2f}")


## この計測結果をどう使うか

課題の種類(add/sub/mul/reverse/count)ごとの正答率と、上の不正解の実例を見てから、
課題セットや温度を変えるかどうかを判断する(**今回はまだ変更しない**)。

変更する場合は、必ず事前登録 `事前登録_内部信号と感情語の接続実験.md` の追記欄に
理由つきで記録すること。
